In [13]:
import pandas as pd
from sklearn.covariance import LedoitWolf
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

In [14]:
returns_10y = pd.read_parquet("data/returns_10y.parquet")
returns_5y  = pd.read_parquet("data/returns_5y.parquet")

esg = pd.read_csv("data/esg.csv", index_col=0)

universe = pd.read_csv("data/universe_djia.csv")
benchmark_weights = pd.read_csv("data/benchmark_weights.csv")


In [15]:
esg = esg.loc[esg['Instrument'] != 'DOW.N'].reset_index(drop=True)

In [16]:
assert (returns_10y.columns == returns_5y.columns).all()
assert (returns_10y.columns == esg.Instrument).all()


In [17]:
def min_sigma(returns):

    # Ledoit–Wolf covariance
    lw = LedoitWolf()
    lw.fit(returns.values)
    Sigma = lw.covariance_

    N = returns.shape[1]

    w = cp.Variable(N)
    prob_minvar = cp.Problem(
        cp.Minimize(cp.quad_form(w, Sigma)),
        [cp.sum(w) == 1, w >= 0]
    )
    prob_minvar.solve(solver=cp.MOSEK)

    w_mv = w.value
    sigma_min = np.sqrt(w_mv @ Sigma @ w_mv) * np.sqrt(252)
    return sigma_min

Step 1: Compute minimum-variance volatility for each horizon

In [18]:

sigma_min_10y = min_sigma(returns_10y)

sigma_min_5y = min_sigma(returns_5y)
print(sigma_min_10y, sigma_min_5y)
sigma_min_common = max(sigma_min_5y, sigma_min_10y)
print(sigma_min_common)


0.13637254168688226 0.15375223769954138
0.15375223769954138


In [19]:
def esg_risk_frontier(returns, esg_vec, sigma_grid, solver=cp.MOSEK):
    """
    Compute ESG–risk efficient frontier.

    returns : pd.DataFrame (T x N)
    esg_vec : np.array (N,)
    sigma_grid : iterable of annual vol targets
    """
    from sklearn.covariance import LedoitWolf
    import numpy as np
    import cvxpy as cp

    # Ledoit–Wolf covariance
    lw = LedoitWolf()
    lw.fit(returns.values)
    Sigma = lw.covariance_

    N = returns.shape[1]
    results = []
    w = cp.Variable(N)

    for sigma_annual in sigma_grid:
        sigma_daily = sigma_annual / np.sqrt(252)
        sigma2 = sigma_daily**2

        w = cp.Variable(N)
        objective = cp.Minimize(-esg_vec @ w)

        constraints = [
            cp.quad_form(w, Sigma) <= sigma2,
            cp.sum(w) == 1,
            w >= 0
        ]

        problem = cp.Problem(objective, constraints)
        problem.solve(solver=solver, verbose=False)

        if problem.status == "optimal":
            w_opt = w.value
            results.append({
                "Volatility": np.sqrt(w_opt @ Sigma @ w_opt) * np.sqrt(252),
                "ESG": esg_vec @ w_opt,
                "Weights": w_opt
            })

    frontier = pd.DataFrame({
        "Volatility": [r["Volatility"] for r in results],
        "ESG": [r["ESG"] for r in results]
    })

    return frontier, results


Step 2: Define a common σ-grid and run optimization:

In [20]:
sigma_grid = np.linspace(0.16, 0.35, 20)
esg_vec = esg["ESG Score"].values
frontier_10y, results_10y = esg_risk_frontier(
    returns_10y,
    esg_vec,
    sigma_grid
)
frontier_5y, results_5y = esg_risk_frontier(
    returns_5y,
    esg_vec,
    sigma_grid
)
import pickle

with open("results/optimized_weights_10y.pkl", "wb") as f:
    pickle.dump(results_10y, f)

with open("results/optimized_weights_5y.pkl", "wb") as f:
    pickle.dump(results_5y, f)


In [21]:
frontier_5y

,Volatility,ESG
0,0.160000,81.577382
1,0.170000,84.534386
2,0.180000,86.083657
3,0.190000,86.991575
4,0.200000,87.645204
5,0.210000,88.091113
6,0.220000,88.456178
7,0.230000,88.778332
8,0.240000,89.073283
9,0.250000,89.316692


In [22]:
frontier_10y

,Volatility,ESG
0,0.16000,86.150217
1,0.17000,87.248569
2,0.18000,87.978642
3,0.19000,88.475899
4,0.20000,88.886053
5,0.21000,89.245562
6,0.22000,89.486087
7,0.23000,89.626225
8,0.24000,89.732821
9,0.25000,89.822700


In [23]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(frontier_10y["Volatility"], frontier_10y["ESG"], marker="o", label="10Y")
plt.plot(frontier_5y["Volatility"], frontier_5y["ESG"], marker="s", label="5Y")

plt.xlabel("Annualized Volatility")
plt.ylabel("Portfolio ESG Score")
plt.title("ESG–Risk Efficient Frontier (DJIA)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("figures/esg_risk_frontier_5y_10y.png", dpi=300)
plt.close()

plt.savefig("figures/esg_risk_frontier_5y_10y.pdf", bbox_inches="tight")
plt.close()



In [25]:
w_bench = pd.read_csv("data/benchmark_weights.csv").set_index("Instrument")['BenchWeight']

# ESG vector aligned to benchmark
esg_aligned = esg.set_index("Instrument").loc[w_bench.index, "ESG Score"]

esg_benchmark = w_bench.values @ esg_aligned.values
print(f"Benchmark ESG score: {esg_benchmark:.2f}")

Benchmark ESG score: 76.55
